In [1]:
import numpy as np
import torch

from torchgpe.bec2D import Gas
from torchgpe.bec2D.potentials import Trap, Contact
from torchgpe.utils.potentials import NonLinearPotential

ModuleNotFoundError: No module named 'torch'

In [ ]:
bec = Gas(N_particles=2e5, grid_size=3e-5)

sigma = 1e-6
bec.psi = torch.exp( -(bec.X**2 + bec.Y**2)/(2*(sigma/bec.adim_length)**2) )

contact = Contact(a_s = 100)
trap = Trap(omegax = 400, omegay = 400)

In [ ]:
import numpy as np
import torch

from torchgpe.bec2D import Gas
from torchgpe.bec2D.potentials import Trap, Contact
from torchgpe.utils.potentials import NonLinearPotential


class DipolarQ2D(NonLinearPotential):
    """
    Quasi-2D dipolar mean-field potential for dipoles polarized along z.

    V_dd(r) = Phi_dd(r)
    with
        Phi_dd(k) = U_dd_2D(k) * n(k)

    Parameters
    ----------
    gdd_2d : float
        Effective 2D dipolar coupling strength in SI units of J*m^2.
        You choose this from your reduction model.
    lz : float
        Harmonic oscillator length of the tightly confined direction (meters).
    """

    def __init__(self, gdd_2d, lz):
        super().__init__()
        self.gdd_2d = gdd_2d
        self.lz = lz

    def on_propagation_begin(self):
        # Build the Fourier-space kernel once.
        # TorchGPE works in adimensional units, so we convert here using gas scales.
        #
        # Coordinates / wavevectors in TorchGPE are adimensionalized by adim_length.
        # Energy is adimensionalized by hbar * adim_pulse.
        #
        # We construct a quasi-2D kernel for polarization along z:
        # U(k) = gdd_2d * F(q),   q = k * lz / sqrt(2)
        #
        # with
        # F(q) = 2 - 3*sqrt(pi)*q*exp(q^2)*erfc(q)
        #
        # This is the common q2D kernel for z-polarized dipoles.
        gas = self.gas
        device = gas.device
        dtype = gas.float_dtype

        # Physical k-grid from adimensional k-grid
        kx = gas.Kx / gas.adim_length
        ky = gas.Ky / gas.adim_length
        k = torch.sqrt(kx**2 + ky**2)

        q = k * self.lz / np.sqrt(2.0)

        # torch.erfc is available in recent PyTorch
        Fq = 2.0 - 3.0 * np.sqrt(np.pi) * q * torch.exp(q**2) * torch.erfc(q)

        # Convert SI coupling to TorchGPE adimensional energy units
        # SI potential energy density term is Phi_dd in joules.
        # Divide by hbar*adim_pulse to express as adimensional potential.
        self._kernel_k = (self.gdd_2d / (gas.hbar * gas.adim_pulse)) * Fq

        # FFT normalization factor:
        # density n2D in SI is |psi|^2 * N / adim_length^2 if psi is normalized to 1
        # Depending on TorchGPE normalization conventions, you may need to absorb an
        # extra factor of N_particles here. This version assumes psi is normalized to 1
        # and uses N_particles explicitly below.
        self._density_prefactor = gas.N_particles / (gas.adim_length**2)

    def potential_function(self, X, Y, psi, time=None):
        # Build physical 2D density from wavefunction
        n2d = torch.abs(psi)**2 * self._density_prefactor

        # Convolution in momentum space
        nk = torch.fft.fft2(n2d)
        phi_dd_si_scaled = torch.fft.ifft2(self._kernel_k * nk).real

        return phi_dd_si_scaled


# -----------------------------
# Example usage
# -----------------------------
bec = Gas(
    N_particles=2e5,
    grid_size=30e-6,                # 30 microns
    adimensionalization_length=1e-6 # 1 micron
)

# Initial guess
sigma = 1.5e-6
bec.psi = torch.exp(
    -(bec.X**2 + bec.Y**2) / (2 * (sigma / bec.adim_length) ** 2)
)

# Harmonic trap in x,y
trap = Trap(omegax=400, omegay=400)

# Optional contact interaction
contact = Contact(a_s=100, a_orth=0.24e-6)

# Example q2D dipolar parameters:
# lz = sqrt(hbar / (m omega_z)) for your transverse confinement
# Here choose something representative, e.g. omega_z = 2*pi*2000 Hz
m_rb87 = 1.44316060e-25
hbar = 1.054571817e-34
omega_z = 2 * np.pi * 2000
lz = np.sqrt(hbar / (m_rb87 * omega_z))

# User-supplied effective q2D dipolar coupling.
# Replace with the value for your species / dipole moment / reduction convention.
gdd_2d = 5.0e-46

dipolar = DipolarQ2D(gdd_2d=gdd_2d, lz=lz)

# Ground state in imaginary time
bec.ground_state(
    potentials=[trap, contact, dipolar],
    N_iterations=int(1e4)
)

# Density after convergence
density = bec.density